Use the repository-root [RUNBOOK](../docs/RUNBOOK.md) for schema-3 reproduction and [verification](../docs/FLIGHTTIME_VERIFICATION.md) for current acceptance. Earlier metrics and retired experiment commands are historical; see the [archive](../evidence/archive/pre-consolidation/INDEX.md). M9 has no new benchmark claim.

# 01 — Airline data exploration

This notebook explores U.S. domestic flight activity and arrival delays from January 2023 through December 2024. It is designed for readers who are new to airline data and Spark.

All large-data work stays in PySpark. Only small grouped tables are converted to Pandas for plotting.

## What does one row represent?

One row represents one scheduled flight operated by a reporting airline on a particular date, from one origin airport to one destination airport. A row contains information known before departure, what actually happened during the flight, and—when applicable—cancellation or delay details.

## Business groups for the columns

The source has many fields. Grouping them by business meaning makes the dataset easier to navigate.

| Group | Main columns | Meaning |
|---|---|---|
| Time | `Year`, `Quarter`, `Month`, `DayofMonth`, `DayOfWeek`, `FlightDate` | Calendar position of the scheduled flight |
| Airline | `Reporting_Airline`, airline IDs/codes, `Tail_Number`, `Flight_Number_Reporting_Airline` | Carrier, aircraft, and flight identity |
| Airports | `Origin`, `Dest`, airport/city/state IDs and names | Where the flight starts and ends |
| Schedule | `CRSDepTime`, `CRSArrTime`, `CRSElapsedTime`, time blocks, `Distance`, `DistanceGroup` | Planned departure, arrival, duration, and route length |
| Actual operations | `DepTime`, `ArrTime`, `WheelsOff`, `WheelsOn`, `TaxiOut`, `TaxiIn`, `ActualElapsedTime`, `AirTime` | Events observed after operations begin |
| Cancellations | `Cancelled`, `CancellationCode` | Whether and broadly why a flight was cancelled |
| Delays | departure/arrival delay fields, delay groups, `Diverted`, and diversion fields | Delay outcomes and irregular operations |
| Delay causes | `CarrierDelay`, `WeatherDelay`, `NASDelay`, `SecurityDelay`, `LateAircraftDelay` | Minutes attributed to standard causes for qualifying delayed flights |

Fields beginning with `Div1` through `Div5` describe individual diversion stops. They are useful for irregular-operations analysis but are not needed for the core charts below.

## 1. Setup and load

The existing Spark runtime supplies the master and workers. Path resolution is internal; only `data/...` is displayed.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
from pyspark import StorageLevel
from pyspark.sql import SparkSession, Window, functions as F

def find_project_root():
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "data/raw/csv").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate data/raw/csv/. Run this notebook inside the Airline project.")

def small_to_pandas(grouped_frame, max_rows=1000):
    """Convert only a deliberately bounded aggregate result for plotting."""
    pdf = grouped_frame.limit(max_rows + 1).toPandas()
    if len(pdf) > max_rows:
        raise ValueError(f"Plot input exceeded the safety limit of {max_rows} aggregated rows.")
    return pdf

PROJECT_ROOT = find_project_root()
DATA_REL = Path("data/raw/csv")
DATA_DIR = PROJECT_ROOT / DATA_REL

spark = SparkSession.builder.appName("airline-01-data-exploration").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

flights = (
    spark.read.option("header", True).option("inferSchema", True).csv(str(DATA_DIR / "*.csv"))
    .withColumn("flight_date", F.to_date("FlightDate"))
    .withColumn(
        "scheduled_departure_hour",
        F.when(F.col("CRSDepTime").cast("int") == 2400, F.lit(0))
        .otherwise(F.floor(F.col("CRSDepTime").cast("int") / 100)),
    )
    .withColumn("route", F.concat_ws(" → ", "Origin", "Dest"))
    .persist(StorageLevel.MEMORY_AND_DISK)
)

print("Spark session ready.")
print(f"Raw input: {DATA_REL.as_posix()}/")

## 2. Dataset overview

Start with scale, date range, and basic business coverage. These values provide context for every later percentage.

In [ ]:
overview = flights.agg(
    F.count("*").alias("flights"),
    F.min("flight_date").alias("first_date"),
    F.max("flight_date").alias("last_date"),
    F.countDistinct("Reporting_Airline").alias("airlines"),
    F.countDistinct("Origin").alias("origin_airports"),
    F.countDistinct("Dest").alias("destination_airports"),
    F.countDistinct("route").alias("routes"),
)
overview.show(vertical=True, truncate=False)

The overview confirms the analytical population and its breadth. Counts below describe rows unless a section explicitly restricts the population to completed flights.

## 3. When do flights operate?

The next three plots show seasonality, weekly patterns, and the daily schedule. They count all scheduled flights, including flights that were later cancelled.

In [ ]:
flights_by_month = (
    flights.groupBy(F.date_trunc("month", "flight_date").alias("month"))
    .agg(F.count("*").alias("flights"))
    .orderBy("month")
)
month_pdf = small_to_pandas(flights_by_month, 24)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(month_pdf["month"], month_pdf["flights"], marker="o")
ax.set(title="Scheduled flights by month", xlabel="Month", ylabel="Flights")
ax.tick_params(axis="x", rotation=45)
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

Monthly changes can reflect seasonality, calendar length, demand, and schedule adjustments. A lower count does not by itself mean worse airline performance.

In [ ]:
weekday_name = (
    F.when(F.col("DayOfWeek") == 1, "Monday")
    .when(F.col("DayOfWeek") == 2, "Tuesday")
    .when(F.col("DayOfWeek") == 3, "Wednesday")
    .when(F.col("DayOfWeek") == 4, "Thursday")
    .when(F.col("DayOfWeek") == 5, "Friday")
    .when(F.col("DayOfWeek") == 6, "Saturday")
    .when(F.col("DayOfWeek") == 7, "Sunday")
)
flights_by_weekday = (
    flights.groupBy("DayOfWeek").agg(F.count("*").alias("flights"))
    .withColumn("weekday", weekday_name).orderBy("DayOfWeek")
)
weekday_pdf = small_to_pandas(flights_by_weekday, 7)

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(weekday_pdf["weekday"], weekday_pdf["flights"], color="#4C78A8")
ax.set(title="Scheduled flights by day of week", xlabel="Day", ylabel="Flights")
ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()

Weekday counts reveal how airlines distribute service across business and leisure travel days.

In [ ]:
flights_by_hour = (
    flights.where(F.col("scheduled_departure_hour").between(0, 23))
    .groupBy("scheduled_departure_hour").agg(F.count("*").alias("flights"))
    .orderBy("scheduled_departure_hour")
)
hour_pdf = small_to_pandas(flights_by_hour, 24)

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(hour_pdf["scheduled_departure_hour"], hour_pdf["flights"], color="#59A14F")
ax.set(title="Scheduled flights by departure hour", xlabel="Scheduled departure hour", ylabel="Flights")
ax.set_xticks(range(0, 24, 2))
plt.tight_layout()
plt.show()

The hourly profile shows the operating day building in the morning and tapering at night. `2400` schedule values are treated as hour 0.

## 4. Airlines, airports, and routes

These rankings show where most records come from. Only the top 20 categories are plotted so labels remain readable; the Spark calculations still use the full dataset.

In [ ]:
top_airlines = (
    flights.groupBy("Reporting_Airline").agg(F.count("*").alias("flights"))
    .orderBy(F.desc("flights")).limit(20)
)
airline_pdf = small_to_pandas(top_airlines, 20).sort_values("flights")

fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(airline_pdf["Reporting_Airline"], airline_pdf["flights"], color="#F28E2B")
ax.set(title="Flights by reporting airline", xlabel="Flights", ylabel="Airline code")
plt.tight_layout()
plt.show()

The carrier mix matters because network structure and operating practices differ across airlines. Counts measure dataset presence, not quality.

In [ ]:
def top_counts(column_name, label_name, n=20):
    return (
        flights.groupBy(F.col(column_name).alias(label_name))
        .agg(F.count("*").alias("flights"))
        .orderBy(F.desc("flights")).limit(n)
    )

ranking_specs = [
    (top_counts("Origin", "category"), "Top origin airports"),
    (top_counts("Dest", "category"), "Top destination airports"),
    (top_counts("route", "category"), "Top routes"),
]

fig, axes = plt.subplots(1, 3, figsize=(18, 7))
for ax, (ranking, title) in zip(axes, ranking_specs):
    pdf = small_to_pandas(ranking, 20).sort_values("flights")
    ax.barh(pdf["category"], pdf["flights"])
    ax.set(title=title, xlabel="Flights", ylabel="")
plt.tight_layout()
plt.show()

Large hubs appear prominently as both origins and destinations. Routes are directional here, so `ATL → MCO` and `MCO → ATL` are separate categories.

## 5. Cancellations

Cancellation rate is cancelled scheduled flights divided by all scheduled flights. It is analyzed separately because a cancelled flight has no arrival-delay outcome.

In [ ]:
cancellation_summary = flights.agg(
    F.count("*").alias("scheduled_flights"),
    F.sum(F.when(F.col("Cancelled") == 1, 1).otherwise(0)).alias("cancelled_flights"),
    F.round(F.avg(F.when(F.col("Cancelled") == 1, 1.0).otherwise(0.0)) * 100, 3).alias("cancellation_rate_pct"),
)
cancellation_summary.show(truncate=False)

cancellation_by_month = (
    flights.groupBy(F.date_trunc("month", "flight_date").alias("month"))
    .agg(F.avg(F.when(F.col("Cancelled") == 1, 1.0).otherwise(0.0)).alias("rate"))
    .withColumn("cancellation_rate_pct", F.round(F.col("rate") * 100, 3))
    .orderBy("month")
)
cancel_month_pdf = small_to_pandas(cancellation_by_month, 24)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(cancel_month_pdf["month"], cancel_month_pdf["cancellation_rate_pct"], marker="o", color="#E15759")
ax.set(title="Cancellation rate by month", xlabel="Month", ylabel="Cancelled flights (%)")
ax.tick_params(axis="x", rotation=45)
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

In [ ]:
cancellation_labels = (
    F.when(F.col("CancellationCode") == "A", "Carrier")
    .when(F.col("CancellationCode") == "B", "Weather")
    .when(F.col("CancellationCode") == "C", "National Air System")
    .when(F.col("CancellationCode") == "D", "Security")
    .otherwise("Missing or other")
)
cancellation_causes = (
    flights.filter(F.col("Cancelled") == 1)
    .withColumn("cancellation_reason", cancellation_labels)
    .groupBy("CancellationCode", "cancellation_reason")
    .agg(F.count("*").alias("cancelled_flights"))
    .orderBy(F.desc("cancelled_flights"))
)
cancellation_causes.show(10, truncate=False)

Monthly spikes often coincide with disruptive events, but the chart alone cannot establish a cause. The cancellation-code table provides a broad attribution category.

## 6. Arrival-delay population

Arrival-delay analysis uses completed, non-diverted flights with a known `ArrDelayMinutes`. A flight is **delayed** when it arrives at least 15 minutes late. Early and on-time arrivals form the non-delayed class.

In [ ]:
completed = (
    flights
    .filter((F.col("Cancelled") == 0) & (F.col("Diverted") == 0) & F.col("ArrDelayMinutes").isNotNull())
    .withColumn("is_delayed_15", (F.col("ArrDelayMinutes") >= 15).cast("int"))
)
completed.agg(
    F.count("*").alias("eligible_completed_flights"),
    F.sum("is_delayed_15").alias("delayed_flights"),
    F.round(F.avg("is_delayed_15") * 100, 3).alias("delay_rate_pct"),
).show(truncate=False)

### Arrival-delay distribution

`ArrDelay` is signed: negative values are early arrivals and positive values are late arrivals. Summary statistics use all eligible flights; the plot is restricted to −60 through 240 minutes so the common range remains visible.

In [ ]:
completed.select("ArrDelay").summary("count", "mean", "stddev", "min", "25%", "50%", "75%", "max").show()

delay_bins = (
    completed.filter(F.col("ArrDelay").between(-60, 240))
    .withColumn("delay_bin_start", (F.floor(F.col("ArrDelay") / 10) * 10).cast("int"))
    .groupBy("delay_bin_start").agg(F.count("*").alias("flights"))
    .orderBy("delay_bin_start")
)
delay_bins_pdf = small_to_pandas(delay_bins, 31)

fig, ax = plt.subplots(figsize=(11, 4))
ax.bar(delay_bins_pdf["delay_bin_start"], delay_bins_pdf["flights"], width=9, color="#4C78A8")
ax.axvline(15, color="#E15759", linestyle="--", label="15-minute threshold")
ax.set(title="Arrival-delay distribution (10-minute bins)", xlabel="Arrival delay in minutes", ylabel="Flights")
ax.legend()
plt.tight_layout()
plt.show()

Most flights cluster near their scheduled arrival, while a long right tail contains severe delays. The bounded chart hides only the visual tail; it does not remove those flights from later rate calculations.

In [ ]:
delay_classes = (
    completed.withColumn("arrival_status", F.when(F.col("is_delayed_15") == 1, "Delayed 15+ min").otherwise("Not delayed"))
    .groupBy("arrival_status").agg(F.count("*").alias("flights"))
    .orderBy(F.desc("flights"))
)
class_pdf = small_to_pandas(delay_classes, 2)

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(class_pdf["arrival_status"], class_pdf["flights"], color=["#59A14F", "#E15759"])
ax.set(title="Delayed versus non-delayed completed flights", xlabel="Arrival status", ylabel="Flights")
plt.tight_layout()
plt.show()

This class balance is important for later machine learning: accuracy alone can be misleading when one outcome is much more common.

## 7. Where and when are delay rates higher?

Each rate is delayed eligible flights divided by all eligible completed flights in that group. Airline, airport, and route charts show the 20 busiest groups, which avoids highlighting tiny samples.

In [ ]:
def delay_rate_by(*group_columns):
    return (
        completed.groupBy(*group_columns)
        .agg(
            F.count("*").alias("flights"),
            F.sum("is_delayed_15").alias("delayed_flights"),
            F.round(F.avg("is_delayed_15") * 100, 3).alias("delay_rate_pct"),
        )
    )

In [ ]:
monthly_delay = delay_rate_by(F.date_trunc("month", "flight_date").alias("period")).orderBy("period")
weekday_delay = delay_rate_by("DayOfWeek").withColumn("weekday", weekday_name).orderBy("DayOfWeek")
hourly_delay = delay_rate_by("scheduled_departure_hour").filter(F.col("scheduled_departure_hour").between(0, 23)).orderBy("scheduled_departure_hour")

monthly_delay_pdf = small_to_pandas(monthly_delay, 24)
weekday_delay_pdf = small_to_pandas(weekday_delay, 7)
hourly_delay_pdf = small_to_pandas(hourly_delay, 24)

fig, axes = plt.subplots(1, 3, figsize=(19, 4))
axes[0].plot(monthly_delay_pdf["period"], monthly_delay_pdf["delay_rate_pct"], marker="o")
axes[0].set(title="Delay rate by month", xlabel="Month", ylabel="Delayed flights (%)")
axes[0].tick_params(axis="x", rotation=45)
axes[1].bar(weekday_delay_pdf["weekday"], weekday_delay_pdf["delay_rate_pct"])
axes[1].set(title="Delay rate by weekday", xlabel="Day", ylabel="Delayed flights (%)")
axes[1].tick_params(axis="x", rotation=45)
axes[2].plot(hourly_delay_pdf["scheduled_departure_hour"], hourly_delay_pdf["delay_rate_pct"], marker="o")
axes[2].set(title="Delay rate by departure hour", xlabel="Scheduled hour", ylabel="Delayed flights (%)")
axes[2].set_xticks(range(0, 24, 2))
plt.tight_layout()
plt.show()

Time patterns can reflect weather seasons, congestion, and delay propagation through the operating day. These are associations, not proof of cause.

In [ ]:
network_delay_specs = [
    (delay_rate_by("Reporting_Airline").orderBy(F.desc("flights")).limit(20), "Reporting_Airline", "Airline"),
    (delay_rate_by("Origin").orderBy(F.desc("flights")).limit(20), "Origin", "Origin airport"),
    (delay_rate_by("Dest").orderBy(F.desc("flights")).limit(20), "Dest", "Destination airport"),
    (delay_rate_by("route").orderBy(F.desc("flights")).limit(20), "route", "Route"),
]

fig, axes = plt.subplots(2, 2, figsize=(16, 14))
for ax, (ranking, category, label) in zip(axes.ravel(), network_delay_specs):
    pdf = small_to_pandas(ranking, 20).sort_values("delay_rate_pct")
    ax.barh(pdf[category], pdf["delay_rate_pct"], color="#B279A2")
    ax.set(title=f"Delay rate among 20 busiest: {label.lower()}", xlabel="Delayed flights (%)", ylabel=label)
plt.tight_layout()
plt.show()

Comparisons are descriptive and should be interpreted alongside volume, geography, schedules, and weather exposure. They are not causal airline or airport rankings.

## 8. Average delay

Two averages answer different questions. Mean `ArrDelay` includes early arrivals as negative minutes; mean `ArrDelayMinutes` floors early arrivals at zero and describes minutes late.

In [ ]:
completed.agg(
    F.round(F.avg("ArrDelay"), 3).alias("average_signed_arrival_delay_minutes"),
    F.round(F.avg("ArrDelayMinutes"), 3).alias("average_minutes_late"),
).show(truncate=False)

average_delay_by_month = (
    completed.groupBy(F.date_trunc("month", "flight_date").alias("month"))
    .agg(
        F.avg("ArrDelay").alias("average_signed_delay"),
        F.avg("ArrDelayMinutes").alias("average_minutes_late"),
    ).orderBy("month")
)
average_pdf = small_to_pandas(average_delay_by_month, 24)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(average_pdf["month"], average_pdf["average_signed_delay"], marker="o", label="Signed arrival delay")
ax.plot(average_pdf["month"], average_pdf["average_minutes_late"], marker="o", label="Minutes late (nonnegative)")
ax.axhline(0, color="black", linewidth=0.8)
ax.set(title="Average arrival delay by month", xlabel="Month", ylabel="Minutes")
ax.tick_params(axis="x", rotation=45)
ax.legend()
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

The two lines should not be compared as if they were interchangeable. Early arrivals pull the signed average downward but do not reduce `ArrDelayMinutes`.

## 9. Delay causes

For flights delayed at least 15 minutes, BTS attributes minutes to five broad causes. Summing those minutes shows system-wide contribution; it does not measure how many distinct flights had each cause.

In [ ]:
cause_columns = ["CarrierDelay", "WeatherDelay", "NASDelay", "SecurityDelay", "LateAircraftDelay"]
delayed_flights = completed.filter(F.col("is_delayed_15") == 1)
cause_totals_wide = delayed_flights.agg(*[
    F.sum(F.coalesce(F.col(column_name), F.lit(0))).alias(column_name)
    for column_name in cause_columns
])
cause_totals = cause_totals_wide.selectExpr(
    "stack(5, 'Carrier', CarrierDelay, 'Weather', WeatherDelay, "
    "'National Air System', NASDelay, 'Security', SecurityDelay, "
    "'Late aircraft', LateAircraftDelay) as (cause, total_minutes)"
).withColumn(
    "share_pct",
    F.round(F.col("total_minutes") / F.sum("total_minutes").over(Window.partitionBy()) * 100, 2),
).orderBy(F.desc("total_minutes"))
cause_pdf = small_to_pandas(cause_totals, 5).sort_values("total_minutes")

fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(cause_pdf["cause"], cause_pdf["total_minutes"], color="#FF9DA7")
ax.set(title="Attributed delay minutes by cause", xlabel="Total attributed minutes", ylabel="Cause")
plt.tight_layout()
plt.show()
cause_totals.show(truncate=False)

Cause fields describe attribution after a delay occurs. They are valuable for diagnosis but would leak the outcome if used to predict delays before departure.

## 10. Cleanup

Release the cached raw DataFrame so cluster memory and disk can be reused.

In [ ]:
flights.unpersist()
spark.stop()
print("Exploration complete; temporary cache released and Spark session stopped.")